# PHLOWER Trajectory Analysis — Treg Compartment

**Project**: P697 TEAseq T1D Low-Dose IL-2

**Date**: 2026-03-30

**Objective**: Test the hypothesis that Treg cells follow an out-and-back trajectory
from Baseline through treatment (Post-IL2 → Post-RAPA) and back to Followup,
using Hodge Laplacian decomposition (PHLOWER).

**Input**: `data/outputData/scenic_export_Treg/` (exported from `P697_DA.rmd`)

**Environment**: `phlower_env` conda environment (see `code/phlower_environment.yml`)

## 0. Setup

In [1]:
import os
import time
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.io
import scipy.sparse
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc

import phlower

import session_info

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (8, 6)

print(f"phlower {phlower.__version__}")
print(f"scanpy {sc.__version__}")
print(f"anndata {ad.__version__}")

phlower 0.1.5
scanpy 1.11.5
anndata 0.11.4


In [2]:
# --- Paths ---
COMP = "Treg"
PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
EXPORT_DIR = os.path.join(PROJECT_DIR, "data", "outputData", f"scenic_export_{COMP}")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "data", "outputData", "phlower")
TABLE_DIR = os.path.join(PROJECT_DIR, "tables")
FIG_DIR = os.path.join(PROJECT_DIR, "figures", "phlower")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

TODAY = datetime.today().strftime("%Y-%m-%d")
print(f"Compartment: {COMP}")
print(f"Export dir:  {EXPORT_DIR}")
print(f"Output dir:  {OUTPUT_DIR}")
print(f"Date stamp:  {TODAY}")

Compartment: Treg
Export dir:  /Users/tedwards/Documents/projects/P697_TEAseq_T1D_low_dose_IL2/data/outputData/scenic_export_Treg
Output dir:  /Users/tedwards/Documents/projects/P697_TEAseq_T1D_low_dose_IL2/data/outputData/phlower
Date stamp:  2026-03-30


## 1. Load Data & Construct AnnData

In [3]:
# --- RNA counts (sparse) ---
rna_counts = scipy.io.mmread(os.path.join(EXPORT_DIR, "RNA_counts.mtx")).T.tocsr()
rna_genes = pd.read_csv(os.path.join(EXPORT_DIR, "RNA_genes.txt"), header=None)[0].values
rna_barcodes = pd.read_csv(os.path.join(EXPORT_DIR, "RNA_barcodes.txt"), header=None)[0].values

print(f"RNA matrix: {rna_counts.shape[0]} cells x {rna_counts.shape[1]} genes")
assert rna_counts.shape == (len(rna_barcodes), len(rna_genes))

RNA matrix: 21664 cells x 27715 genes


In [4]:
# --- Metadata ---
meta = pd.read_csv(os.path.join(EXPORT_DIR, "metadata.csv"), index_col=0)
meta.index.name = None

# Rename Washout → Followup if needed (scenic_export may predate rename)
if "Washout" in meta["treatmentPhase"].values:
    meta["treatmentPhase"] = meta["treatmentPhase"].replace("Washout", "Followup")
    print("Renamed Washout → Followup")

# Ensure consistent ordering
meta = meta.loc[rna_barcodes]

# Diagnostic: cells per donor × treatment phase
print("\nCells per donor × treatmentPhase:")
ct = pd.crosstab(meta["donorID"], meta["treatmentPhase"])
print(ct)
print(f"\nTotal cells: {len(meta)}")

# Flag any donors missing Baseline
if "Baseline" in ct.columns:
    missing_bl = ct.index[ct["Baseline"] == 0].tolist()
    if missing_bl:
        print(f"\n⚠ Donor(s) missing Baseline: {missing_bl} — their cells won't be roots")


Cells per donor × treatmentPhase:
treatmentPhase  Baseline  Followup  Post-IL2  Post-RAPA
donorID                                                
1                   1098      1103       622        702
2                      0       647       384        457
3                   1007       929       774        419
5                    866       896       519        193
6                    330       348       263        332
7                    832       973       481        519
8                   1274      1109       654        931
9                   1161       998       679        164

Total cells: 21664

⚠ Donor(s) missing Baseline: [2] — their cells won't be roots


In [5]:
# --- Embeddings ---
emb_harmony = pd.read_csv(os.path.join(EXPORT_DIR, "embedding_harmony.csv"), index_col=0)
emb_umap = pd.read_csv(os.path.join(EXPORT_DIR, "embedding_umap.wnn.csv"), index_col=0)

# Use first 18 harmony dims (matching WNN configuration)
N_HARMONY_DIMS = 18  # TODO: TUNABLE — number of harmony dims used for WNN RNA component
harmony_arr = emb_harmony.loc[rna_barcodes].values[:, :N_HARMONY_DIMS]
umap_arr = emb_umap.loc[rna_barcodes].values

print(f"Harmony embedding: {harmony_arr.shape} (using first {N_HARMONY_DIMS} of {emb_harmony.shape[1]})")
print(f"UMAP embedding:    {umap_arr.shape}")

Harmony embedding: (21664, 18) (using first 18 of 30)
UMAP embedding:    (21664, 2)


In [6]:
# --- Construct AnnData ---
adata = ad.AnnData(
    X=rna_counts,
    obs=meta,
    var=pd.DataFrame(index=rna_genes),
)
adata.obsm["X_pca"] = harmony_arr      # PHLOWER expects 'X_pca' by default
adata.obsm["X_umap"] = umap_arr

# Ensure categorical types
adata.obs["treatmentPhase"] = pd.Categorical(
    adata.obs["treatmentPhase"],
    categories=["Baseline", "Post-IL2", "Post-RAPA", "Followup"],
    ordered=True,
)
adata.obs["wnn_clusters"] = adata.obs["wnn_clusters"].astype(str).astype("category")
adata.obs["donorID"] = adata.obs["donorID"].astype(str).astype("category")

print(adata)
print(f"\nobsm keys: {list(adata.obsm.keys())}")

AnnData object with n_obs × n_vars = 21664 × 27715
    obs: 'donorID', 'visitNum', 'treatmentPhase', 'wnn_clusters', 'RNA.weight', 'ATAC.weight'
    obsm: 'X_pca', 'X_umap'

obsm keys: ['X_pca', 'X_umap']


## 2. DDHodge Pseudotime

In [7]:
# --- Define root cells (Baseline) ---
roots = (adata.obs["treatmentPhase"] == "Baseline").values.tolist()
n_roots = sum(roots)
print(f"Root cells (Baseline): {n_roots} / {adata.n_obs} ({100*n_roots/adata.n_obs:.1f}%)")

Root cells (Baseline): 6568 / 21664 (30.3%)


In [ ]:
# --- DDHodge ---
# TODO: TUNABLE PARAMETERS
#   k    : kNN bandwidth (default 11, higher = smoother graph)
#   npc  : number of PCs to use (we use N_HARMONY_DIMS harmony components)
#   ndc  : number of diffusion components (default 40)
#   s    : diffusion time scale (default 1)

t0 = time.time()
print(f"Starting ddhodge at {datetime.now().strftime('%H:%M:%S')}...")

phlower.ext.ddhodge(
    adata,
    basis="X_pca",
    roots=roots,
    k=11,               # TODO: TUNABLE
    npc=N_HARMONY_DIMS, # TODO: TUNABLE — match the harmony dims
    ndc=40,             # TODO: TUNABLE
    s=1,                # TODO: TUNABLE
)

elapsed = time.time() - t0
print(f"ddhodge completed in {elapsed/60:.1f} min")

Starting ddhodge at 15:56:39...
2026-03-30 15:56:39.951632 distance_matrix
2026-03-30 15:57:01.531254 Diffusionmaps: 
done.
2026-03-30 15:59:33.104214 diffusion distance:
2026-03-30 16:00:30.627600 transition matrix:
2026-03-30 16:01:01.907869 graph from A
2026-03-30 16:01:03.261760 Rewiring: 
2026-03-30 16:01:03.261851 div(g_o)...
2026-03-30 16:03:49.416312 edge weight...
2026-03-30 16:03:50.868545 cholesky solve ax=b...
+ 1e-06 I is positive-definite


: 

## 3. Pseudotime Diagnostics

In [ ]:
# Graph layout colored by treatment phase
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

phlower.pl.nxdraw_group(
    adata,
    group_name="treatmentPhase",
    show_edges=True,
    ax=axes[0],
    s=5,
)
axes[0].set_title("Graph — Treatment Phase")

phlower.pl.nxdraw_group(
    adata,
    group_name="wnn_clusters",
    show_edges=True,
    ax=axes[1],
    s=5,
)
axes[1].set_title("Graph — WNN Clusters")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_graph_layout.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Graph colored by pseudotime
fig, ax = plt.subplots(1, 1, figsize=(8, 6))
phlower.pl.nxdraw_score(adata, color="u", ax=ax, s=5)
ax.set_title("Graph — Pseudotime (u)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_pseudotime_graph.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Pseudotime distribution by treatment phase
# Extract pseudotime from graph nodes → cells
phlower.tl.assign_graph_node_attr_to_adata(adata, attr_name="u")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# By treatment phase
phase_order = ["Baseline", "Post-IL2", "Post-RAPA", "Followup"]
sns.violinplot(
    data=adata.obs, x="treatmentPhase", y="u",
    order=phase_order, ax=axes[0], inner="box",
)
axes[0].set_title("Pseudotime by Treatment Phase")
axes[0].set_xlabel("")
axes[0].tick_params(axis='x', rotation=45)

# By WNN cluster
cluster_order = sorted(adata.obs["wnn_clusters"].unique(), key=lambda x: int(x) if x.isdigit() else x)
sns.boxplot(
    data=adata.obs, x="wnn_clusters", y="u",
    order=cluster_order, ax=axes[1],
)
axes[1].set_title("Pseudotime by WNN Cluster")
axes[1].set_xlabel("")

# By donor
sns.boxplot(
    data=adata.obs, x="donorID", y="u",
    ax=axes[2],
)
axes[2].set_title("Pseudotime by Donor")
axes[2].set_xlabel("")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_pseudotime_distributions.pdf"), bbox_inches="tight")
plt.show()

## 4. Delaunay Triangulation

In [ ]:
# TODO: TUNABLE PARAMETERS
#   start_n       : number of start nodes (default 5)
#   end_n         : number of end nodes (default 5)
#   circle_quant  : quantile for circle detection (default 0.1)
#   trunc_quantile: truncation quantile for long edges (default 0.75)

t0 = time.time()
print(f"Starting Delaunay triangulation at {datetime.now().strftime('%H:%M:%S')}...")

phlower.tl.construct_delaunay(
    adata,
    cluster_name="wnn_clusters",
    node_attr="u",
    start_n=10,          # TODO: TUNABLE
    end_n=10,            # TODO: TUNABLE
    circle_quant=0.1,    # TODO: TUNABLE
    trunc_quantile=0.75, # TODO: TUNABLE
    calc_layout=True,
)

elapsed = time.time() - t0
print(f"Delaunay completed in {elapsed:.1f} s")

In [ ]:
# Visualize Delaunay graph with holes
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

phlower.pl.nxdraw_holes(
    adata,
    ax=axes[0],
)
axes[0].set_title("Delaunay — Holes (potential cycles)")

phlower.pl.nxdraw_group(
    adata,
    graph_name="Delaunay_graph",
    layout_name="Delaunay_layout",
    group_name="wnn_clusters",
    ax=axes[1],
    s=30,
)
axes[1].set_title("Delaunay — WNN Clusters")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_delaunay.pdf"), bbox_inches="tight")
plt.show()

## 5. Hodge Laplacian Decomposition

In [ ]:
t0 = time.time()
print(f"Starting L1 Hodge Laplacian decomposition at {datetime.now().strftime('%H:%M:%S')}...")

phlower.tl.L1Norm_decomp(adata)

elapsed = time.time() - t0
print(f"L1Norm_decomp completed in {elapsed:.1f} s")

In [ ]:
# Knee point detection for eigenvalues
phlower.tl.knee_eigen(adata, plot=True)

plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_knee_eigen.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Eigenvalue line plot — KEY READOUT
# Eigenvalues ≈ 0 indicate harmonic components (cycles)
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
phlower.pl.plot_eigen_line(adata, n_eig=20, ax=ax)
ax.set_title(f"{COMP} — L1 Eigenvalues (harmonic ≈ 0 → cycles)")
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='zero')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_eigenvalues.pdf"), bbox_inches="tight")
plt.show()

## 6. Random Walk Trajectories

In [ ]:
# TODO: TUNABLE PARAMETERS
#   n : number of random walk trajectories (default 10000)

t0 = time.time()
print(f"Starting random walk at {datetime.now().strftime('%H:%M:%S')}...")

phlower.tl.random_climb_knn(
    adata,
    n=10000,  # TODO: TUNABLE
)

elapsed = time.time() - t0
print(f"Random walk completed in {elapsed:.1f} s")

## 7. Trajectory Projection & Clustering

In [ ]:
t0 = time.time()
print(f"Starting trajectory matrix at {datetime.now().strftime('%H:%M:%S')}...")

phlower.tl.trajs_matrix(adata)

elapsed = time.time() - t0
print(f"Trajectory matrix completed in {elapsed:.1f} s")

In [ ]:
# TODO: TUNABLE — eps for DBSCAN clustering
phlower.tl.trajs_clustering(
    adata,
    eps=0.5,  # TODO: TUNABLE
)

# Show trajectory cluster sizes
if "trajs_clusters" in adata.uns:
    clusters = adata.uns["trajs_clusters"]
    from collections import Counter
    counts = Counter(clusters)
    print("Trajectory cluster sizes:")
    for k, v in sorted(counts.items()):
        print(f"  Cluster {k}: {v} trajectories")

In [ ]:
# Trajectory lines in harmonic space (2D)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

phlower.pl.plot_trajectory_harmonic_lines(
    adata,
    dims=[0, 1],
    ax=axes[0],
    sample_ratio=0.1,
)
axes[0].set_title(f"{COMP} — Trajectory Lines (dims 0-1)")

phlower.pl.plot_trajectory_harmonic_points(
    adata,
    dims=[0, 1],
    ax=axes[1],
    sample_ratio=0.1,
)
axes[1].set_title(f"{COMP} — Trajectory Points (dims 0-1)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_trajectories_harmonic.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Trajectory embedding (dimensionality reduction of trajectory space)
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

phlower.pl.plot_trajs_embedding(
    adata,
    ax=ax,
)
ax.set_title(f"{COMP} — Trajectory Clusters (embedding)")

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_trajs_embedding.pdf"), bbox_inches="tight")
plt.show()

## 8. Stream Tree

In [ ]:
# TODO: TUNABLE PARAMETERS
#   min_bin_number : minimum cells per bin (default 5)
#   cut_threshold  : threshold for cutting branches (default 1)

t0 = time.time()
print(f"Starting harmonic stream tree at {datetime.now().strftime('%H:%M:%S')}...")

phlower.tl.harmonic_stream_tree(
    adata,
    cluster="wnn_clusters",
    min_bin_number=20,  # TODO: TUNABLE
    cut_threshold=1,    # TODO: TUNABLE
)

elapsed = time.time() - t0
print(f"Stream tree completed in {elapsed:.1f} s")

In [ ]:
# Stream tree embedding
fig, ax = plt.subplots(1, 1, figsize=(10, 8))
phlower.pl.plot_stream_tree_embedding(adata, ax=ax)
ax.set_title(f"{COMP} — Stream Tree")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_stream_tree.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Stream plot colored by treatment phase
phlower.ext.plot_stream_sc(
    adata,
    color=["treatmentPhase"],
    fig_size=(10, 6),
)
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_stream_sc_treatmentPhase.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Stream plot colored by WNN clusters
phlower.ext.plot_stream_sc(
    adata,
    color=["wnn_clusters"],
    fig_size=(10, 6),
)
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_stream_sc_wnn_clusters.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Smooth stream plot by treatment phase
phlower.ext.plot_stream(
    adata,
    color=["treatmentPhase"],
    fig_size=(10, 6),
)
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_stream_treatmentPhase.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# Smooth stream plot by WNN clusters
phlower.ext.plot_stream(
    adata,
    color=["wnn_clusters"],
    fig_size=(10, 6),
)
plt.savefig(os.path.join(FIG_DIR, f"phlower_{COMP}_stream_wnn_clusters.pdf"), bbox_inches="tight")
plt.show()

## 9. Save Results

In [ ]:
# Save full AnnData as pickle
pickle_path = os.path.join(OUTPUT_DIR, f"phlower_{COMP}_{TODAY}.pickle")
with open(pickle_path, "wb") as f:
    pickle.dump(adata, f)
print(f"Saved: {pickle_path}")

# Export pseudotime + trajectory assignments as CSV
export_df = adata.obs[["donorID", "visitNum", "treatmentPhase", "wnn_clusters"]].copy()
if "u" in adata.obs.columns:
    export_df["pseudotime_u"] = adata.obs["u"]

csv_path = os.path.join(TABLE_DIR, f"phlower_{COMP}_{TODAY}_pseudotime.csv")
export_df.to_csv(csv_path)
print(f"Saved: {csv_path}")

## 10. Session Info

In [ ]:
session_info.show()